In [1]:
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import CLIPProcessor, CLIPModel

In [2]:
# Load CLIP model and processor
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [3]:
# Load GPT-2 model and tokenizer
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
# Freeze CLIP and GPT-2
for param in clip_model.parameters():
    param.requires_grad = False
for param in gpt2_model.parameters():
    param.requires_grad = False

In [5]:
# Define the mapping network
class MappingNetwork(nn.Module):
    def __init__(self, clip_embedding_dim, gpt2_embedding_dim, prefix_size):
        super(MappingNetwork, self).__init__()
        self.prefix_size = prefix_size
        self.transformer = nn.Transformer(
            d_model=gpt2_embedding_dim, 
            nhead=8, 
            num_encoder_layers=8,
        )
        self.fc = nn.Linear(clip_embedding_dim, gpt2_embedding_dim)

    def forward(self, clip_embedding):
        # Project CLIP embeddings to GPT-2 embedding size
        projected = self.fc(clip_embedding)
        # Repeat to match prefix size and apply transformer
        repeated = projected.unsqueeze(1).repeat(1, self.prefix_size, 1)
        output = self.transformer(repeated, repeated)  # Self-attention
        return output

In [13]:
# Parameters
# clip_embedding_dim = clip_model.visual.output_dim
clip_embedding_dim = 768
gpt2_embedding_dim = gpt2_model.config.hidden_size
prefix_size = 10

(clip_embedding_dim, gpt2_embedding_dim, prefix_size)

(768, 768, 10)

In [14]:
# Instantiate mapping network
mapping_network = MappingNetwork(clip_embedding_dim, gpt2_embedding_dim, prefix_size)
optimizer = torch.optim.Adam(mapping_network.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()


C:\Users\Hanish\AppData\Roaming\Python\Python311\site-packages\torch\nn\modules\transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [15]:
# Dataset and training loop
from torch.utils.data import DataLoader
from torchvision.datasets import CocoCaptions
from torchvision.transforms import Compose, Resize, ToTensor

In [ ]:
# Define dataset and data loader
transform = Compose([Resize((224, 224)), ToTensor()])
# train_dataset = CocoCaptions(root='../VQA/Images/mscoco/train2014',
#                              annFile='../VQA/Annotations/v2_mscoco_train2014_annotations.json',
#                              transform=transform)
# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

loading annotations into memory...
Done (t=8.61s)
creating index...


KeyError: 'id'

In [4]:
import datasets as ds

dataset = ds.load_dataset(
    "shunk031/MSCOCO",
    year=2014,
    coco_task="captions",
    trust_remote_code=True,
)


Generating train split: 0 examples [00:00, ? examples/s]

Load images:   0%|          | 0/82782 [00:00<?, ?it/s]

Load captions data:   0%|          | 0/414113 [00:00<?, ?it/s]

DatasetGenerationError: An error occurred while generating the dataset

In [1]:
train_dataset = dataset
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

NameError: name 'dataset' is not defined

In [ ]:

# Training
for epoch in range(10):
    for images, captions in train_loader:
        # Get CLIP embeddings
        clip_inputs = clip_processor(images=images, return_tensors="pt")
        clip_outputs = clip_model.get_image_features(**clip_inputs)
        
        # Map embeddings to GPT-2 space
        prefix_embeddings = mapping_network(clip_outputs)
        
        # Tokenize and shift captions for causal modeling
        tokenized_captions = gpt2_tokenizer(captions, return_tensors="pt", padding=True, truncation=True)
        input_ids = tokenized_captions["input_ids"]
        labels = input_ids.clone()
        
        # Prefix embeddings + captions as input to GPT-2
        outputs = gpt2_model(inputs_embeds=prefix_embeddings, labels=labels)
        loss = outputs.loss
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch + 1}, Loss: {loss.item()}")

# Save trained model
torch.save(mapping_network.state_dict(), "mapping_network.pth")


In [23]:
import json

cat = 'train'
dir = "C:/Users/Hanish/.cache/huggingface/datasets/downloads/extracted/da3bf8bd551ed3c2d58b53f61522659cab1980791ed332ad176da4e3ca5bc134/annotations/"
json_file = f"captions_{cat}2014.json"
with open(dir + json_file, 'r') as file:
    data = json.load(file)
# print(data)  

In [14]:
data.keys()

dict_keys(['info', 'images', 'licenses', 'annotations'])

In [15]:
data['annotations'][:5]

[{'image_id': 318556,
  'id': 48,
  'caption': 'A very clean and well decorated empty bathroom'},
 {'image_id': 116100,
  'id': 67,
  'caption': 'A panoramic view of a kitchen and all of its appliances.'},
 {'image_id': 318556,
  'id': 126,
  'caption': 'A blue and white bathroom with butterfly themed wall tiles.'},
 {'image_id': 116100,
  'id': 148,
  'caption': 'A panoramic photo of a kitchen and dining room'},
 {'image_id': 379340,
  'id': 173,
  'caption': 'A graffiti-ed stop sign across the street from a red car '}]

In [16]:
data['images'][:5]

[{'license': 5,
  'file_name': 'COCO_train2014_000000057870.jpg',
  'coco_url': 'http://images.cocodataset.org/train2014/COCO_train2014_000000057870.jpg',
  'height': 480,
  'width': 640,
  'date_captured': '2013-11-14 16:28:13',
  'flickr_url': 'http://farm4.staticflickr.com/3153/2970773875_164f0c0b83_z.jpg',
  'id': 57870},
 {'license': 5,
  'file_name': 'COCO_train2014_000000384029.jpg',
  'coco_url': 'http://images.cocodataset.org/train2014/COCO_train2014_000000384029.jpg',
  'height': 429,
  'width': 640,
  'date_captured': '2013-11-14 16:29:45',
  'flickr_url': 'http://farm3.staticflickr.com/2422/3577229611_3a3235458a_z.jpg',
  'id': 384029},
 {'license': 1,
  'file_name': 'COCO_train2014_000000222016.jpg',
  'coco_url': 'http://images.cocodataset.org/train2014/COCO_train2014_000000222016.jpg',
  'height': 640,
  'width': 480,
  'date_captured': '2013-11-14 16:37:59',
  'flickr_url': 'http://farm2.staticflickr.com/1431/1118526611_09172475e5_z.jpg',
  'id': 222016},
 {'license': 3

In [17]:
len(data['images'])

82783

In [18]:
files_in_ann = []
for i in range(len(data['images'])):
    files_in_ann.append(data['images'][i]['file_name'])

In [19]:
len(files_in_ann)

82783

In [27]:
import os

img_dir = f"C:/Users/Hanish/.cache/huggingface/datasets/downloads/extracted/ba405d943cc4bfcfdc027b9b9190c713492605b0bfcfa2c2aa491e0078d58a0a/train2014"
images = os.listdir(path=img_dir)

In [28]:
len(images)

71136